# 16 · Completed AIA 72-hour saved-model verification

**Completed on 4 October 2026 at 16:04 UTC.** Models 29 and 43 each reproduced all 3,905 earlier model-selection predictions within the declared tolerances, with zero mismatches. Model 17's earlier accepted replay is reused. The average uses all three models' saved probabilities.

All six executed code cells and their outputs below are unchanged from the VM archive. Local verification checked the original comparison rows, the three-model probability average, notebook execution and the final controller/cleanup receipts. The replay took 433.59 seconds; the full charged reservation took 444.29 seconds, with zero overrun.

This establishes saved-model reproducibility. Later-period forecasting, calibration, uncertainty and SHARP–AIA fusion remain pending. Candidate labels and historical availability retain their documented limitations.

[Verified result and provenance](../results/aia72_completion_replay_completed_20261004/summary.json)

## Context & Methods
The comparison uses the original 3,905 January–June 2014 selection cases, original train-only normalization, six channels and three histories. Predictions must match within the predeclared `atol=1e-6, rtol=1e-5`. Batches contain 16 cases, with four loader workers retaining the persistent lease. A reviewed invocation is bounded to 30 minutes including setup, with 60 seconds reserved for cleanup. Actual work stops promptly after completion.

### Key assumptions and limits
This tests saved-model reproducibility using the same architecture/preprocessing implementation. It is not independent scientific validation, later-period evaluation, calibration or fusion. Labels and historical availability remain provisional.

### 1. Setup and exact input binding
Use the repository and its pinned artifacts. Dependencies are NumPy, pandas, PyTorch and nbformat; GPU execution additionally requires the exact recorded VM runtime. Normal execution previews metadata. The reviewed controller supplies GPU context after completed-training verification.

In [1]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
import torch

if 'REPLAY_CONTEXT' not in globals():
    candidates=[Path.cwd(),*Path.cwd().parents]
    ROOT=next(p for p in candidates if (p/'scripts/aia72_completion_replay.py').is_file())
    REPLAY_CONTEXT={'mode':'preview','bundle_root':str(ROOT)}
ROOT=Path(REPLAY_CONTEXT['bundle_root'])
sys.path.insert(0,str(ROOT))
from scripts.aia72_replay_contract import require
from scripts.aia72_replay import before_deadline
from scripts.aia72_completion_replay import load_completed_support
from scripts.aia72_completion_replay_contract import check_contract
configuration=ROOT/'configs/aia72_completion_replay_v1.json'
CONTRACT=check_contract(json.loads(configuration.read_text())) if configuration.is_file() else None
display(pd.DataFrame([{'Mode':REPLAY_CONTEXT['mode'],'Inputs pinned':CONTRACT is not None,
                      'New model checks':'29,43','Prior check reused':17,'Maximum reservation minutes':30}]))

Mode,Inputs pinned,New model checks,Prior check reused,Maximum reservation minutes
authorized_completed_model_replay,True,"29,43",17,30


## Data
### 2. Verify completed models and preserved support
The metadata check hashes inputs and verifies selected epochs, case identities, labels, normalization support and the first model's accepted receipt. It does not read image pixels. Missing final checkpoints remain a visible preparation dependency.

In [2]:
if CONTRACT is None:
    print('Final input binding pending completed training and its verified archive. No image pixels or GPU model run.')
else:
    source=REPLAY_CONTEXT.get('preview_source_root',CONTRACT['source_root'])
    paths,selected,references,completions,records,normalization=load_completed_support(CONTRACT,source)
    display(pd.DataFrame([{'Seed':seed,'Selected epoch':completions[seed]['best_epoch'],
                          'Selection cases':len(references[seed]),'Replay needed':seed!=17}
                         for seed in [17,29,43]]))
    print('Completed-model inputs and earlier-selection support verified. Image pixels read: 0.')

Seed,Selected epoch,Selection cases,Replay needed
17,2,3905,False
29,3,3905,True
43,3,3905,True


Completed-model inputs and earlier-selection support verified. Image pixels read: 0.


### 3. Read-only prediction loop
This is the executable inference function used by the worker's hash-verified module. It checks ordered identities and labels, unchanged parameters/buffers, the deadline and resource guards. It constructs no optimizer.

In [3]:
def infer_rows(model, batches, expected, device, deadline, progress=None, guard=None):
    """Independent inference loop: fixed order, no optimizer, unchanged model buffers."""
    model.eval()
    original = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    rows = []
    if guard:
        guard()
    with torch.no_grad():
        for images, labels, identities in batches:
            before_deadline(deadline)
            if guard:
                guard()
            offset = len(rows)
            require(list(identities) == expected.forecast_case_id.iloc[offset:offset + len(identities)].tolist(),
                    'Loader omitted/reordered/duplicated cases')
            require(np.array_equal(labels.numpy(), expected.label.iloc[offset:offset + len(labels)].to_numpy()),
                    'Loader outcomes changed')
            values = model(images.to(device, non_blocking=True)).detach().cpu().numpy()
            require(values.shape == (len(identities),) and np.isfinite(values).all(), 'Invalid replay logits')
            rows.extend({'forecast_case_id': identity, 'label': int(label), 'logit': float(value)}
                        for identity, label, value in zip(identities, labels, values))
            if progress:
                progress(len(rows))
            before_deadline(deadline)
            if guard:
                guard()
    require(len(rows) == len(expected), 'Incomplete replay support')
    require(all(torch.equal(original[k], v.detach().cpu()) for k, v in model.state_dict().items()),
            'Inference mutated parameters or buffers')
    return pd.DataFrame(rows)


### 4. Keep every seed in the ensemble
After successful checks, average the three saved probability vectors on identical cases. Averaging logits or dropping an inconvenient seed would change the prespecified experiment.

In [4]:
def selection_ensemble(references):
    require(set(references) == {17, 29, 43}, 'All three fixed seeds are required')
    first = references[17]
    require(len(first) > 0 and first.forecast_case_id.is_unique, 'Empty or duplicate ensemble support')
    require(first.label.isin([0, 1]).all(), 'Unknown ensemble outcomes')
    result = first[['forecast_case_id', 'label']].copy()
    for seed in [17, 29, 43]:
        frame = references[seed]
        require(frame.forecast_case_id.tolist() == first.forecast_case_id.tolist(), 'Ensemble support/order changed')
        require(np.array_equal(frame.label, first.label), 'Ensemble labels changed')
        logits = frame.logit.to_numpy()
        require(np.isfinite(logits).all(), 'Nonfinite ensemble logits')
        result[f'probability_seed_{seed}'] = 1 / (1 + np.exp(-np.clip(logits, -700, 700)))
    result['probability'] = result[[f'probability_seed_{s}' for s in [17, 29, 43]]].mean(axis=1)
    return result


## Results
### 5. Execute only under the reviewed controller
The model and loader processes share the inherited persistent lock. Any input change, prediction mismatch, stop or incomplete cleanup prevents an accepted result. A notebook preview is not GPU execution.

In [5]:
if REPLAY_CONTEXT['mode']=='authorized_completed_model_replay':
    require(CONTRACT is not None,'Final inputs are not pinned')
    from scripts.aia72_completion_replay_worker import run
    result=run(CONTRACT,REPLAY_CONTEXT)
else:
    require(REPLAY_CONTEXT['mode']=='preview','Unknown notebook mode')
    result={'status':'metadata_preview_no_scientific_replay','fitting_steps':0,
            'scientific_acceptance':False,'comparisons':[]}
display(pd.DataFrame([{'Status':result['status'],'Fitting steps':result['fitting_steps'],
                      'Independent scientific validation':result['scientific_acceptance']}]))
if result['comparisons']:
    display(pd.DataFrame(result['comparisons'])[['seed','best_epoch','cases','mismatched_cases','max_absolute_logit_difference']])

Status,Fitting steps,Independent scientific validation
technical_three_seed_replay_verified_pending_controller,0,False


seed,best_epoch,cases,mismatched_cases,max_absolute_logit_difference
29,3,3905,0,8.881784e-16
43,3,3905,0,8.881784e-16


## Takeaways
### 6. Preserve the exact status
The final controller receipt establishes technical acceptance and cleanup. The executed notebook, per-case comparisons and three-seed ensemble stay together. Gray-Box scheduling priority continues through the remaining later-period and fusion analysis.

In [6]:
print('Recorded notebook status:',result['status'])
print('Later AIA inference, probability calibration, uncertainty and SHARP/AIA fusion are separate phases.')

Recorded notebook status: technical_three_seed_replay_verified_pending_controller
Later AIA inference, probability calibration, uncertainty and SHARP/AIA fusion are separate phases.
